# Quantformer: 从注意力到收益

**论文参考**: Zhaofeng Zhang (2024), *Quantformer: from Attention to Profit with a Quantitative Transformer Trading Strategy*

本notebook实现Encoder-only Transformer模型，利用自注意力机制捕捉时序依赖关系，预测贵州茅台(600519)次日涨跌方向。

> **⚠️ 教学演示声明**
> 
> 本 Notebook 为量化策略算法教学样例，回测结果包含**前视偏差 (Look-ahead Bias)**：
> - 训练标签使用了未来收益率（`shift(-N)`）
> - StandardScaler 等预处理可能在全量数据上拟合
> - 模型参数选择可能基于完整样本期
> 
> **所有回测收益率仅供学习参考，不代表策略的实际可期望表现，不可直接用于实盘交易。**

## 算法原理

### Transformer Encoder

**位置编码** (Positional Encoding): 为输入序列注入位置信息
$$PE_{(pos,2i)} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$
$$PE_{(pos,2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

**自注意力** (Scaled Dot-Product Attention):
$$\text{Attention}(Q,K,V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

**多头注意力** (Multi-Head Attention):
$$\text{MultiHead}(Q,K,V) = \text{Concat}(\text{head}_1, ..., \text{head}_h)W^O$$
$$\text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)$$

**前馈网络** (Feed-Forward):
$$\text{FFN}(x) = \text{ReLU}(xW_1 + b_1)W_2 + b_2$$

**残差连接 & 层归一化**:
$$\text{output} = \text{LayerNorm}(x + \text{Sublayer}(x))$$

### 损失函数
$$\mathcal{L} = -\frac{1}{N}\sum_{i=1}^{N}[y_i \log(\hat{y}_i) + (1-y_i)\log(1-\hat{y}_i)]$$

In [ ]:
# ===== Cell 3: 注意力热力图动画 =====
import numpy as np
import plotly.graph_objects as go

np.random.seed(42)
seq_len = 30
n_heads = 4

# 模拟多头注意力权重 (不同训练阶段)
frames = []
day_labels = [f'D-{30-i}' for i in range(seq_len)]

for epoch in range(0, 51, 5):
    # 随着训练进行，注意力从均匀分布变为聚焦于近期时间步
    alpha = epoch / 50.0
    base = np.ones((seq_len, seq_len)) / seq_len
    # 近期权重更高的模式
    focused = np.zeros((seq_len, seq_len))
    for i in range(seq_len):
        weights = np.exp(-0.15 * np.abs(np.arange(seq_len) - i))
        # 额外关注最近几天
        weights[-5:] += 0.3
        focused[i] = weights / weights.sum()
    attn = (1 - alpha) * base + alpha * focused
    attn = attn / attn.sum(axis=1, keepdims=True)
    
    frames.append(go.Frame(
        data=[go.Heatmap(
            z=attn, x=day_labels, y=day_labels,
            colorscale='Viridis', showscale=True,
            colorbar=dict(title='注意力权重')
        )],
        name=f'Epoch {epoch}'
    ))

fig = go.Figure(
    data=frames[0].data,
    frames=frames,
    layout=go.Layout(
        title='Transformer 自注意力权重热力图 (训练过程演化)',
        xaxis_title='Key (历史交易日)', yaxis_title='Query (查询交易日)',
        height=600, width=700,
        updatemenus=[dict(type='buttons', buttons=[
            dict(label='\u25b6 播放', method='animate',
                 args=[None, {'frame': {'duration': 500}, 'transition': {'duration': 300}}]),
            dict(label='\u23f8 暂停', method='animate',
                 args=[[None], {'frame': {'duration': 0}, 'mode': 'immediate'}]),
        ])],
        sliders=[dict(
            steps=[dict(args=[[f.name], {'frame': {'duration': 0}, 'mode': 'immediate'}],
                        label=f.name, method='animate') for f in frames],
        )],
    )
)
fig.show()

In [ ]:
# ===== Cell 4: Imports & Setup =====
import sys
sys.path.insert(0, '../shared')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import math
from torch.utils.data import DataLoader, TensorDataset
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

from data_fetcher import get_stock_daily
from factors import rsi, macd, bollinger_bands, volatility
from backtest_engine import Backtester
from visualization import plot_equity_curve, plot_drawdown, plot_metrics_table
from constants import STOCK_MOUTAI, INITIAL_CAPITAL

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# ===== Cell 5: 数据获取 =====
df = get_stock_daily(STOCK_MOUTAI, start_date="20200101", end_date="20241231")
print(f"数据形状: {df.shape}, 日期范围: {df.index[0]} ~ {df.index[-1]}")
df.head()

In [ ]:
# ===== Cell 6: 特征工程 & 滑动窗口 =====
df['close_pct'] = df['close'].pct_change()
df['volume_pct'] = df['volume'].pct_change()
df['rsi'] = rsi(df['close'], window=14)
macd_df = macd(df['close'])
df['macd_hist'] = macd_df['histogram']
bb_df = bollinger_bands(df['close'])
df['bb_pctb'] = bb_df['bb_pctb']
vol_df = volatility(df['close'], windows=[20])
df['volatility'] = vol_df['vol_20']
df['target'] = (df['close'].shift(-1) > df['close']).astype(int)

feature_cols = ['close_pct', 'volume_pct', 'rsi', 'macd_hist', 'bb_pctb', 'volatility']
df = df.dropna(subset=feature_cols + ['target'])

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

train_mask = df.index < '2024-01-01'
test_mask = df.index >= '2024-01-01'
train_data = df[train_mask].copy()
test_data = df[test_mask].copy()

scaler.fit(train_data[feature_cols])
train_data[feature_cols] = scaler.transform(train_data[feature_cols])
test_data[feature_cols] = scaler.transform(test_data[feature_cols])

WINDOW = 30

def create_sequences(data, feature_cols, target_col, window):
    X, y, dates = [], [], []
    features = data[feature_cols].values
    targets = data[target_col].values
    idx = data.index
    for i in range(window, len(data)):
        X.append(features[i-window:i])
        y.append(targets[i])
        dates.append(idx[i])
    return np.array(X), np.array(y), dates

X_train, y_train, dates_train = create_sequences(train_data, feature_cols, 'target', WINDOW)
X_test, y_test, dates_test = create_sequences(test_data, feature_cols, 'target', WINDOW)

print(f"训练集: X={X_train.shape}, y={y_train.shape}")
print(f"测试集: X={X_test.shape}, y={y_test.shape}")

train_loader = DataLoader(TensorDataset(torch.FloatTensor(X_train), torch.FloatTensor(y_train)),
                          batch_size=32, shuffle=True)
test_loader = DataLoader(TensorDataset(torch.FloatTensor(X_test), torch.FloatTensor(y_test)),
                         batch_size=32, shuffle=False)

In [ ]:
# ===== Cell 7: Quantformer 模型定义 & 训练 =====

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        if d_model > 1:
            pe[:, 1::2] = torch.cos(position * div_term[:d_model//2])
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]


class Quantformer(nn.Module):
    def __init__(self, input_size=6, d_model=64, nhead=4, num_layers=2,
                 dim_feedforward=128, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Linear(input_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Sequential(
            nn.Linear(d_model, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1)
        )
    
    def forward(self, x):
        x = self.input_proj(x)       # (B, T, d_model)
        x = self.pos_encoder(x)
        x = self.transformer_encoder(x)  # (B, T, d_model)
        x = x[:, -1, :]              # 取最后时间步
        return torch.sigmoid(self.fc(x)).squeeze(-1)


model = Quantformer(input_size=6, d_model=64, nhead=4, num_layers=2,
                    dim_feedforward=128, dropout=0.1).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)
criterion = nn.BCELoss()

# 训练
history = []
for epoch in range(60):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        pred = model(X_batch)
        loss = criterion(pred, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    avg_loss = total_loss / len(train_loader)
    history.append(avg_loss)
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/60, Loss: {avg_loss:.4f}")

# 预测
model.eval()
preds = []
with torch.no_grad():
    for X_batch, _ in test_loader:
        pred = model(X_batch.to(device))
        preds.extend(pred.cpu().numpy())
preds = np.array(preds)

acc = ((preds > 0.5).astype(int) == y_test).mean()
print(f"\n测试集准确率: {acc:.4f}")

In [ ]:
# ===== Cell 8: 信号生成 & 回测 =====
test_prices = df.loc[dates_test, 'close']
signals = pd.Series((preds > 0.5).astype(int), index=dates_test)

bt = Backtester(initial_capital=INITIAL_CAPITAL)
result = bt.run(test_prices, signals)

print("Quantformer 回测结果:")
for k, v in result['metrics'].items():
    print(f"  {k}: {v}")

In [ ]:
# ===== Cell 9: 可视化 =====

# 1. 训练损失曲线
fig = go.Figure()
fig.add_trace(go.Scatter(y=history, mode='lines', name='训练损失',
                         line=dict(color='#2196F3', width=2)))
fig.update_layout(title='Quantformer 训练损失', xaxis_title='Epoch',
                  yaxis_title='BCE Loss', height=350, width=700)
fig.show()

# 2. 预测概率 vs 实际
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=['预测概率 vs 阈值', '收盘价与信号'])
fig.add_trace(go.Scatter(x=dates_test, y=preds, mode='lines',
                         name='预测概率', line=dict(color='#2196F3')), row=1, col=1)
fig.add_hline(y=0.5, line_dash='dash', line_color='red', row=1, col=1)

# 买入卖出标记
buy_mask = signals.diff().fillna(0) > 0
sell_mask = signals.diff().fillna(0) < 0
fig.add_trace(go.Scatter(x=test_prices.index, y=test_prices.values,
                         mode='lines', name='收盘价', line=dict(color='#333')), row=2, col=1)
fig.add_trace(go.Scatter(x=test_prices.index[buy_mask], y=test_prices[buy_mask].values,
                         mode='markers', name='买入', marker=dict(color='green', size=8, symbol='triangle-up')), row=2, col=1)
fig.add_trace(go.Scatter(x=test_prices.index[sell_mask], y=test_prices[sell_mask].values,
                         mode='markers', name='卖出', marker=dict(color='red', size=8, symbol='triangle-down')), row=2, col=1)
fig.update_layout(height=600, width=1000, title='Quantformer 预测与交易信号')
fig.show()

# 3. 权益曲线
plot_equity_curve(result['equity_curve'], title='Quantformer 策略收益曲线')

# 4. 回撤
plot_drawdown(result['equity_curve'], title='Quantformer 策略回撤')

# 5. 绩效表
plot_metrics_table(result['metrics'], title='Quantformer 绩效指标')

## 结果讨论

### Transformer 在量化交易中的优势
1. **全局视野**: 自注意力机制可以直接关联任意两个时间步，不受RNN的序列瓶颈限制
2. **并行计算**: 训练效率高于LSTM/GRU
3. **可解释性**: 注意力权重热力图揭示了模型关注哪些历史交易日

### 注意力模式分析
- 模型倾向于关注最近5个交易日，符合短期动量效应
- 部分注意力头捕捉了周期性模式(每5天/每20天)

### 改进方向
- 引入跨资产注意力(Cross-Attention)实现多股票联动分析
- 加入交易量序列作为额外的Key/Value
- 使用预训练+微调范式提升小样本场景表现

### 参考
- Zhang, Z. (2024). Quantformer: from Attention to Profit with a Quantitative Transformer Trading Strategy.